# Práctica 3 · UrbanIng-V2X

**Objetivo:** detectar vehículos, generar sus bounding boxes 3D y contar los presentes en la intersección.

Se utilizan únicamente los LiDAR de infraestructura **11, 12, 31 y 32**. **Sin tracking:** cada fotograma se procesa de forma independiente.

## Imports

In [ ]:
import base64
import csv
import hashlib
import html
import json
import urllib.error
import urllib.request
import xml.etree.ElementTree as ET
from contextlib import ExitStack
from dataclasses import dataclass
from functools import cache, cached_property
from pathlib import Path
from typing import ClassVar

import numpy as np
import matplotlib.pyplot as plt
import multivolumefile
import py7zr
from matplotlib.figure import Figure
from PIL import Image, ImageDraw, ImageFont
from pyproj import Transformer
from scipy.ndimage import gaussian_filter, median_filter
from scipy.spatial import cKDTree
from shapely import intersects_xy
from shapely.geometry import LineString, MultiPoint, Polygon, box as rectangle, mapping
from shapely.ops import unary_union

from IPython.display import HTML, display

# Estilo común de las figuras
plt.rcParams.update({'figure.dpi': 115, 'font.size': 10, 'axes.titlesize': 12})

## Parámetros

In [ ]:
# Rutas (el notebook se ejecuta desde la carpeta del proyecto)
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'                  # carpeta de descarga del devkit urbaning
DATASET_DIR = DATA_DIR / 'dataset'
MAP_FILE = DATA_DIR / 'crossings_lanelet2map.osm'
CONFIG_FILE = ROOT / 'detector_config.json'
RESULTS_DIR = ROOT / 'results' / 'improved'
VIDEOS_DIR = ROOT / 'results' / 'videos'
assert CONFIG_FILE.is_file(), 'Para reejecutar, abrir el notebook desde la carpeta del proyecto.'

# Secuencias y LiDAR de infraestructura
SEQUENCES = ('20241126_0024_crossing1_09',
             '20241126_0008_crossing1_01',
             '20241127_0000_crossing1_00')
SENSORS = tuple(f'crossing1_{i}_lidar' for i in (11, 12, 31, 32))
GPS_ORIGIN = (11.438043, 48.771731)  # lon, lat del origen de crossing1 (devkit urbaning)

# Parámetros de preprocesamiento
ROI = (-45, 45, -45, 45)      # xmin, xmax, ymin, ymax en metros (coordenadas globales)
VOXEL_SIZE = 0.15             # tamaño del vóxel en metros
FOREGROUND_MAX_HEIGHT = 4.5   # altura máxima sobre el suelo de un punto de vehículo (m)
ROAD_MARGIN = 0.35            # margen de la máscara vial (m)
ROAD_CONTEXT_MARGIN = 0.5     # margen extra para no cortar carrocerías en el borde (m)

# Parámetros del detector: valores por defecto sobrescritos por detector_config.json
# (la misma combinación que usa la CLI)
DETECTOR_DEFAULTS = {'eps': 0.85, 'min_points': 4, 'min_height': 0.4,
                     'min_length': 1.0, 'min_width': 0.3, 'complete_boxes': True}
DETECTOR_CONFIG = {**DETECTOR_DEFAULTS,
                   **json.loads(CONFIG_FILE.read_text(encoding='utf-8'))['parameters']}

# Parámetros de presentación
DEMO_SEQUENCE = SEQUENCES[1]
DEMO_FRAME = 100
FRAMES_PER_SEQUENCE = 200
VIDEO_FPS = 10                # debe coincidir con la frecuencia de los LiDAR (10 Hz), que fija el vídeo

# Exportación de resultados (mismo valor por defecto que main.py)
PREVIEW_EVERY = 20            # vista cenital PNG cada 20 fotogramas (0 la desactiva)

## Carga y fusión de los LiDAR

In [ ]:
@dataclass
class LidarFrame:
    sequence: str
    index: int
    timestamp_ms: int
    sources: dict   # sensor -> archivo sincronizado
    clouds: dict    # sensor -> nube en coordenadas globales

    @cached_property
    def points(self):
        # Las cuatro nubes se fusionan antes de detectar para no duplicar vehículos.
        return np.concatenate(list(self.clouds.values()))


class SequenceDataset:
    """Fotogramas sincronizados de una secuencia, fusionados en coordenadas globales."""

    def __init__(self, name, root=DATASET_DIR):
        self.name = name
        self.folder = Path(root) / name
        self.calibration = json.loads((self.folder / 'calibration.json').read_text())
        self.sync = self.read_sync(self.folder / 'timesync_info.csv')

    def __len__(self):
        return len(self.sync['timestamp_ms'])

    @property
    def timestamps_ms(self):
        return np.asarray(self.sync['timestamp_ms'], dtype=np.int64)

    def frame(self, index):
        sources = {sensor: self.filename(sensor, index) for sensor in SENSORS}
        clouds = {sensor: self.sensor_cloud(sensor, index) for sensor in SENSORS}
        return LidarFrame(self.name, index, int(self.sync['timestamp_ms'][index]), sources, clouds)

    def sensor_cloud(self, sensor, index):
        """Nube de un LiDAR en coordenadas globales, sin puntos no finitos."""
        path = self.folder / sensor / self.filename(sensor, index)
        with np.load(path, allow_pickle=False) as data:
            xyz = np.column_stack([data[k] for k in ('x', 'y', 'z')])
        xyz = xyz[np.isfinite(xyz).all(axis=1)]
        return self.transform(xyz, self.calibration[sensor]['extrinsics']['gTl'])

    def filename(self, sensor, index):
        filename = self.sync[sensor][index]
        # Solo nombres simples: evita leer fuera de la carpeta del sensor.
        if not filename or Path(filename).name != filename:
            raise ValueError(f'Archivo inválido: {sensor}, frame {index}')
        return filename

    @staticmethod
    def read_sync(path):
        """timesync_info.csv: cada fila es una columna -> archivos o tiempos por fotograma."""
        with Path(path).open(newline='') as stream:
            return {row[0]: row[1:] for row in csv.reader(stream)}

    @staticmethod
    def transform(points, matrix):
        """p_global = R · p_LiDAR + t (convención de LidarData en el devkit)."""
        matrix = np.asarray(matrix, dtype=float)
        if matrix.shape != (4, 4) or not np.isfinite(matrix).all():
            raise ValueError('Calibración inválida')
        return points @ matrix[:3, :3].T + matrix[:3, 3]

## Mapa vial

In [ ]:
class RoadMap:
    """Calzada y ejes de carril del mapa lanelet2 oficial; no usa anotaciones de objetos."""
    AXIS_STEP = 2       # separación aproximada entre muestras del eje de carril (m)
    AXIS_RADIUS = 100   # solo se conservan ejes cercanos al cruce (m)

    def __init__(self, path=MAP_FILE, roi=ROI, margin=ROAD_MARGIN):
        # El OSM se lee una sola vez para la calzada y para los ejes de carril.
        self.lanelets = self._read_lanelets(path)
        self.extent = rectangle(roi[0], roi[2], roi[1], roi[3])
        self.polygon = self._road_polygon(margin)

    @cached_property
    def context(self):
        """Calzada ampliada para no cortar la carrocería al borde de la calzada."""
        return self.polygon.buffer(ROAD_CONTEXT_MARGIN)

    @cached_property
    def lane_segments(self):
        """Segmentos [x0, y0, x1, y1] de los ejes de carril."""
        segments = []
        for left, right in self.lanelets:
            axis = self._lane_axis(left, right)
            for start, end in zip(axis[:-1], axis[1:]):
                near = np.linalg.norm((start + end) / 2) < self.AXIS_RADIUS
                if np.linalg.norm(end - start) > 0.01 and near:
                    segments.append([*start, *end])
        return np.array(segments)

    def inside(self, points, area=None):
        points = np.asarray(points)
        return intersects_xy(self.polygon if area is None else area, points[:, 0], points[:, 1])

    def on_road(self, boxes):
        """Máscara de las cajas cuyo centro cae sobre la calzada."""
        if not boxes:
            return np.zeros(0, dtype=bool)
        return self.inside(np.array([b['center'][:2] for b in boxes]))

    def lane_heading(self, xy):
        """Dirección del eje de carril más cercano al punto xy."""
        start, end = self.lane_segments[:, :2], self.lane_segments[:, 2:]
        direction = end - start
        t = np.clip(np.sum((xy - start) * direction, axis=1) / np.sum(direction ** 2, axis=1), 0, 1)
        nearest = np.argmin(np.linalg.norm(start + t[:, None] * direction - xy, axis=1))
        return np.arctan2(direction[nearest, 1], direction[nearest, 0])

    @cached_property
    def rings(self):
        """Contornos exteriores e interiores de la calzada (ya recortada a la ROI)."""
        polygons = list(self.polygon.geoms) if hasattr(self.polygon, 'geoms') else [self.polygon]
        return [ring for polygon in polygons for ring in (polygon.exterior, *polygon.interiors)]

    def draw(self, ax):
        for ring in self.rings:
            ax.plot(*ring.xy, color='tab:blue', linewidth=0.9)

    def _road_polygon(self, margin):
        polygons = [Polygon(np.concatenate((left, right[::-1]))).buffer(0)
                    for left, right in self.lanelets]
        polygons = [polygon for polygon in polygons if polygon.intersects(self.extent)]
        if not polygons:
            raise ValueError('No hay carriles viales dentro de la ROI')
        return unary_union(polygons).buffer(margin).intersection(self.extent)

    @classmethod
    def _lane_axis(cls, left, right):
        """Puntos medios entre ambos bordes del carril, cada ~2 m."""
        a, b = LineString(left), LineString(right)
        count = max(2, int(max(a.length, b.length) / cls.AXIS_STEP) + 1)
        return np.array([(np.array(a.interpolate(t, normalized=True).coords[0])
                          + np.array(b.interpolate(t, normalized=True).coords[0])) / 2
                         for t in np.linspace(0, 1, count)])

    @staticmethod
    def _read_lanelets(path):
        """Bordes (izquierdo, derecho) de los lanelets viales, en UTM32 relativo al origen (m)."""
        root = ET.parse(path).getroot()
        project = Transformer.from_crs('EPSG:4326', 'EPSG:32632', always_xy=True)
        origin = np.array(project.transform(*GPS_ORIGIN))
        nodes = {n.get('id'): np.array(project.transform(float(n.get('lon')), float(n.get('lat'))))
                 - origin for n in root.findall('node')}
        ways = {w.get('id'): np.array([nodes[n.get('ref')] for n in w.findall('nd')])
                for w in root.findall('way')}
        lanelets = []
        for relation in root.findall('relation'):
            tags = {t.get('k'): t.get('v') for t in relation.findall('tag')}
            if tags.get('type') != 'lanelet' or tags.get('subtype') != 'road':
                continue
            bounds = {m.get('role'): ways[m.get('ref')] for m in relation.findall('member')
                      if m.get('type') == 'way' and m.get('role') in ('left', 'right')}
            left, right = bounds['left'], bounds['right']
            # Ambos bordes deben recorrerse en el mismo sentido.
            if np.linalg.norm(left[0] - right[0]) > np.linalg.norm(left[0] - right[-1]):
                right = right[::-1]
            lanelets.append((left, right))
        return lanelets

## Suelo y filtrado de la nube

In [ ]:
@dataclass
class GroundModel:
    """Plano global z = ax + by + c (RANSAC) corregido con una rejilla local de 3 m."""
    CELL: ClassVar[float] = 3.0   # lado de la celda de corrección local (m)

    plane: np.ndarray
    grid: np.ndarray
    origin: np.ndarray

    @classmethod
    def fit(cls, points):
        plane = cls._ransac_plane(points)
        return cls(plane, *cls._local_grid(points, plane))

    def height(self, points):
        """Altura sobre el suelo local de los puntos usados en el ajuste."""
        cells = self._cells(points[:, :2], self.origin)
        return self.residual(points, self.plane) - self.grid[tuple(cells.T)]

    def bottom(self, xy):
        """Cota del suelo bajo un centro XY (se usa la celda más próxima de la rejilla)."""
        index = np.clip(self._cells(xy, self.origin), 0, np.array(self.grid.shape) - 1)
        return float(np.r_[xy, 1] @ self.plane + self.grid[tuple(index)])

    @classmethod
    def _cells(cls, xy, origin):
        return np.floor((xy - origin) / cls.CELL).astype(int)

    @staticmethod
    def residual(points, plane):
        """Altura sobre el plano global z = ax + by + c, sin corrección local."""
        return points[:, 2] - points[:, :2] @ plane[:2] - plane[2]

    @staticmethod
    def _ransac_plane(points):
        """RANSAC sobre los mínimos de celdas XY de 2 m (semilla fija, reproducible)."""
        if len(points) < 30:
            raise ValueError('No hay suficientes puntos para estimar el suelo')
        _, groups = np.unique(np.floor(points[:, :2] / 2).astype(int), axis=0, return_inverse=True)
        order = np.lexsort((points[:, 2], groups))
        low = points[order[np.r_[True, np.diff(groups[order]) != 0]]]
        if len(low) < 10:
            raise ValueError('Cobertura insuficiente para estimar el suelo')
        design = np.column_stack((low[:, :2], np.ones(len(low))))
        rng = np.random.default_rng(7)
        best = np.zeros(len(low), dtype=bool)
        for _ in range(150):
            ids = rng.choice(len(low), 3, replace=False)
            if np.linalg.matrix_rank(design[ids]) != 3:
                continue
            model = np.linalg.solve(design[ids], low[ids, 2])
            if np.linalg.norm(model[:2]) > 0.15:   # pendiente máxima admitida
                continue
            inliers = np.abs(design @ model - low[:, 2]) < 0.18
            if inliers.sum() > best.sum():
                best = inliers
        if best.sum() < 10:
            raise ValueError('No se ha podido estimar el suelo')
        return np.linalg.lstsq(design[best], low[best, 2], rcond=None)[0]

    @classmethod
    def _local_grid(cls, points, plane):
        """Mínimo residuo por celda, relleno por vecino más cercano y suavizado."""
        origin = np.floor(points[:, :2].min(axis=0) / cls.CELL) * cls.CELL
        indices = cls._cells(points[:, :2], origin)
        residual = cls.residual(points, plane)
        grid = np.full(tuple(indices.max(axis=0) + 1), np.inf)
        # Solo cotas cercanas al plano inicial, para no tomar vehículos como suelo.
        valid = (residual > -0.4) & (residual < 0.3)
        np.minimum.at(grid, tuple(indices[valid].T), residual[valid])
        known = np.isfinite(grid)
        if not known.any():
            grid[:] = 0
            return grid, origin
        locations = np.column_stack(np.nonzero(known))
        missing = np.column_stack(np.nonzero(~known))
        if len(missing):
            nearest = cKDTree(locations).query(missing)[1]
            grid[tuple(missing.T)] = grid[tuple(locations[nearest].T)]
        return gaussian_filter(median_filter(grid, size=3), sigma=1), origin


@dataclass
class FilteredCloud:
    """Etapas intermedias del filtrado de un fotograma."""
    cropped: np.ndarray      # puntos dentro de la ROI
    reduced: np.ndarray      # un punto por vóxel
    foreground: np.ndarray   # puntos elevados próximos a la calzada
    ground: GroundModel


class PointCloudFilter:
    """ROI, vóxeles, suelo y máscara vial; lo comparten la demostración y el detector."""

    def __init__(self, road, min_height, roi=ROI, voxel=VOXEL_SIZE):
        self.road = road
        self.roi = roi
        self.voxel = voxel
        self.min_height = min_height

    def apply(self, points):
        cropped = self.crop(points, self.roi)
        reduced = self.voxelize(cropped, self.voxel)
        ground = GroundModel.fit(reduced)
        height = ground.height(reduced)
        # No se exige movimiento: los vehículos detenidos también se conservan.
        mask = ((height >= self.min_height) & (height <= FOREGROUND_MAX_HEIGHT)
                & self.road.inside(reduced, self.road.context))
        return FilteredCloud(cropped, reduced, reduced[mask], ground)

    @staticmethod
    def crop(points, roi):
        xmin, xmax, ymin, ymax = roi
        return points[(points[:, 0] >= xmin) & (points[:, 0] <= xmax)
                      & (points[:, 1] >= ymin) & (points[:, 1] <= ymax)]

    @staticmethod
    def voxelize(points, size):
        """Un punto por vóxel (el primero), conservando el orden original."""
        if not len(points):
            return points
        _, indices = np.unique(np.floor(points / size).astype(np.int64), axis=0, return_index=True)
        return points[np.sort(indices)]

## Agrupamiento y ajuste de cajas

In [ ]:
class XYClusterer:
    """DBSCAN en el plano XY con vecinos calculados bajo demanda."""

    def __init__(self, eps, min_points):
        self.eps = eps
        self.min_points = min_points

    def groups(self, points):
        if not len(points):
            return
        tree = cKDTree(points[:, :2])
        core = tree.query_ball_point(points[:, :2], self.eps, return_length=True) >= self.min_points
        labels = np.full(len(points), -1, dtype=int)
        cluster_id = 0
        for seed in np.flatnonzero(core):
            if labels[seed] >= 0:
                continue
            labels[seed] = cluster_id
            stack = [seed]
            while stack:
                current = stack.pop()
                neighbors = np.asarray(tree.query_ball_point(points[current, :2], self.eps))
                new = neighbors[labels[neighbors] < 0]
                labels[new] = cluster_id
                stack.extend(new[core[new]].tolist())
            yield points[labels == cluster_id]
            cluster_id += 1


class BoxFitter:
    """Caja orientada de un clúster, con base en el suelo local y completado de caras ocultas."""
    MIN_DIMENSIONS = (4.0, 1.8)   # largo y ancho mínimos al completar (m)
    SENSOR_MASTS = np.array([[-20.27, -2.59], [18.30, 1.46]])   # mástiles de los LiDAR (XY global)
    PLACEMENTS = ('away', 'centered', 'opposite')

    def __init__(self, ground, road=None, complete=True):
        self.ground = ground
        self.road = road   # RoadMap: su eje de carril orienta los clústeres pequeños
        self.complete = complete

    def fit(self, points, yaw=None, placement='away'):
        hull = MultiPoint(points[:, :2]).minimum_rotated_rectangle
        if hull.geom_type != 'Polygon':
            return None
        if yaw is None:
            yaw = self._heading(hull, points)
        rotation = self.rotation(yaw)
        local = points[:, :2] @ rotation
        lo, hi = local.min(axis=0), local.max(axis=0)
        observed = hi - lo
        center = (lo + hi) / 2
        observed_center = center @ rotation.T
        dimensions = observed.copy()
        if self.complete:
            # Priorización conservadora para carrocerías parcialmente visibles.
            dimensions = np.maximum(dimensions, self.MIN_DIMENSIONS)
            center = self._place(center, rotation, dimensions, observed, placement)
        xy = center @ rotation.T
        bottom = self.ground.bottom(xy)
        height = float(np.percentile(points[:, 2], 98) - bottom)
        return {'center': [float(xy[0]), float(xy[1]), bottom + height / 2],
                'dimensions': [float(dimensions[0]), float(dimensions[1]), height],
                'observed_dimensions_xy': observed.tolist(),
                'observed_center_xy': observed_center.tolist(),
                'yaw_rad': float((yaw + np.pi / 2) % np.pi - np.pi / 2),
                'class': 'vehicle_candidate', 'num_points': len(points)}

    def _heading(self, hull, points):
        """Lado largo del rectángulo mínimo; con poco soporte, dirección del carril."""
        corners = np.array(hull.exterior.coords)[:4]
        edges = np.roll(corners, -1, axis=0) - corners
        lengths = np.linalg.norm(edges, axis=1)
        edge = edges[np.argmax(lengths)]
        yaw = np.arctan2(edge[1], edge[0])
        if self.road is not None and (lengths.max() < 3.2 or lengths.min() < 0.8):
            yaw = self.road.lane_heading(points[:, :2].mean(axis=0))
        return yaw

    def _place(self, center, rotation, dimensions, observed, placement):
        """Desplaza la caja completada respecto al mástil más cercano (cara oculta)."""
        xy = center @ rotation.T
        nearest = np.argmin(np.linalg.norm(self.SENSOR_MASTS - xy, axis=1))
        viewpoint = self.SENSOR_MASTS[nearest] @ rotation
        shift = np.sign(center - viewpoint) * (dimensions - observed) / 2
        if placement == 'away':
            return center + shift
        if placement == 'opposite':
            return center - shift
        if placement != 'centered':
            raise ValueError('Colocación de caja desconocida')
        return center

    @staticmethod
    def rotation(yaw):
        return np.array([[np.cos(yaw), -np.sin(yaw)], [np.sin(yaw), np.cos(yaw)]])

    @classmethod
    def footprint(cls, box):
        """Huella BEV orientada de una caja métrica."""
        corners = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]])
        half = np.array(box['dimensions'][:2]) / 2
        return Polygon(corners * half @ cls.rotation(box['yaw_rad']).T + box['center'][:2])

## Fusión de fragmentos y detector

In [ ]:
@dataclass
class Fragment:
    """Grupo de puntos, su caja y cuántos clústeres DBSCAN lo forman."""
    points: np.ndarray
    box: dict
    count: int = 1


class FragmentResolver:
    """Fusión espacial de fragmentos y completado de cajas sin invadir las de más evidencia.

    Solo usa puntos del instante actual: no hay etiquetas ni asociaciones temporales.
    """

    def __init__(self, fitter, config):
        self.fitter = fitter
        self.config = config

    def resolve(self, groups):
        """Fragmentos aceptados, cada uno con su caja final."""
        boxes = map(self.fitter.fit, groups)
        fragments = [Fragment(group, box) for group, box in zip(groups, boxes) if box is not None]
        if self.config.get('merge_fragments', True):
            fragments = self._merge(fragments)
        kept, polygons = [], []
        for fragment in sorted(fragments, key=lambda f: f.box['num_points'], reverse=True):
            original = fragment.box
            if not self.plausible(original):
                continue
            chosen = self._resolve_overlap(fragment.points, original, polygons)
            polygon = BoxFitter.footprint(chosen)
            # El solapamiento fuerte restante se considera una hipótesis duplicada.
            if self.is_duplicate(polygon, polygons, self.config.get('nms_overlap', 0.5)):
                continue
            chosen['fragments_merged'] = fragment.count
            chosen['overlap_resolved'] = chosen is not original
            kept.append(Fragment(fragment.points, chosen, fragment.count))
            polygons.append(polygon)
        return kept

    def plausible(self, box):
        """Dimensiones observadas compatibles con un vehículo."""
        length, width = box['observed_dimensions_xy']
        return (self.config['min_length'] <= length <= 18
                and self.config['min_width'] <= width <= 3.5
                and 0.65 <= box['dimensions'][2] <= 4.5)

    @staticmethod
    def is_duplicate(polygon, polygons, overlap):
        return any(polygon.intersection(q).area / min(polygon.area, q.area) > overlap
                   for q in polygons)

    @staticmethod
    def _axis_angle(a, b):
        """Diferencia entre los ejes de dos cajas, en [0, π/2] (el sentido no importa)."""
        delta = 2 * (a['yaw_rad'] - b['yaw_rad'])
        return abs(np.arctan2(np.sin(delta), np.cos(delta))) / 2

    def _merge(self, fragments):
        """Une parejas compatibles hasta que no quede ninguna (siempre la primera encontrada)."""
        while (merge := self._find_merge(fragments)) is not None:
            i, j, fragment = merge
            fragments[i] = fragment
            del fragments[j]
        return fragments

    def _find_merge(self, fragments):
        for i in range(len(fragments)):
            for j in range(i + 1, len(fragments)):
                merged = self._try_merge(fragments[i], fragments[j])
                if merged is not None:
                    return i, j, merged
        return None

    def _try_merge(self, first, second):
        a, b = first.box, second.box
        if first.count + second.count > 3:
            return None
        if self._axis_angle(a, b) > np.deg2rad(25):
            return None
        pa, pb = BoxFitter.footprint(a), BoxFitter.footprint(b)
        if pa.intersection(pb).area / min(pa.area, pb.area) < 0.12:
            return None
        if cKDTree(first.points[:, :2]).query(second.points[:, :2])[0].min() > 1.8:
            return None
        combined = np.concatenate((first.points, second.points))
        merged = self.fitter.fit(combined)
        if merged is None:
            return None
        length, width = merged['observed_dimensions_xy']
        if length > 7.0 or width > 2.8:
            return None
        # La unión debe explicar un único vehículo, sin gran superficie vacía.
        if BoxFitter.footprint(merged).area > 1.3 * (pa.area + pb.area):
            return None
        return Fragment(combined, merged, first.count + second.count)

    def _resolve_overlap(self, group, original, polygons):
        """Elige la colocación de menor coste si la caja invade otras ya aceptadas."""
        polygon = BoxFitter.footprint(original)
        if not self.config.get('resolve_overlap', True) or not any(
                polygon.intersection(q).area > 0.15 for q in polygons):
            return original
        proposals = [(0.0, original), *self._proposals(group, original)]
        return min(proposals, key=lambda proposal: self._cost(proposal, polygons))[1]

    def _proposals(self, group, original):
        yaws = [original['yaw_rad']]
        # Con poco soporte, el carril más cercano puede tener la dirección perpendicular.
        if max(original['observed_dimensions_xy']) < 3.2:
            yaws.append(original['yaw_rad'] + np.pi / 2)
        for yaw in yaws:
            for placement in BoxFitter.PLACEMENTS:
                candidate = self.fitter.fit(group, yaw, placement)
                if candidate is None or candidate['dimensions'][1] > 3.5:
                    continue
                shift = np.linalg.norm(np.array(candidate['center'][:2]) - original['center'][:2])
                yield 0.05 * shift + (0.12 if yaw != original['yaw_rad'] else 0), candidate

    @staticmethod
    def _cost(proposal, polygons):
        prior, box = proposal
        polygon = BoxFitter.footprint(box)
        return prior + 10 * sum(polygon.intersection(q).area for q in polygons)


class VehicleDetector:
    """Detección geométrica por fotograma: DBSCAN XY, cajas orientadas y refinamiento espacial."""

    def __init__(self, road, config=DETECTOR_CONFIG):
        self.road = road
        self.config = config
        self.clusterer = XYClusterer(self.config['eps'], self.config['min_points'])

    def detect(self, cloud):
        """Candidatos aceptados: cada Fragment une la caja final y los puntos de su clúster."""
        groups = list(self.clusterer.groups(cloud.foreground))
        # El ajuste depende del suelo de cada fotograma.
        fitter = BoxFitter(cloud.ground, self.road, self.config['complete_boxes'])
        resolver = FragmentResolver(fitter, self.config)
        if self.config.get('spatial_refinement', True):
            return self._on_road(resolver.resolve(groups))
        candidates = [Fragment(group, box) for group, box in zip(groups, map(fitter.fit, groups))
                      if box is not None and resolver.plausible(box)]
        return self._suppress_duplicates(self._on_road(candidates))

    def _on_road(self, candidates):
        """Conserva los candidatos cuyo centro cae sobre la calzada."""
        mask = self.road.on_road([c.box for c in candidates])
        return [c for c, valid in zip(candidates, mask) if valid]

    @staticmethod
    def _suppress_duplicates(candidates, overlap=0.5):
        """Supresión de cajas duplicadas por fragmentos; no usa identidades temporales."""
        kept, polygons = [], []
        for candidate in sorted(candidates, key=lambda c: c.box['num_points'], reverse=True):
            polygon = BoxFitter.footprint(candidate.box)
            if FragmentResolver.is_duplicate(polygon, polygons, overlap):
                continue
            kept.append(candidate)
            polygons.append(polygon)
        return kept

## Pipeline de procesamiento

In [ ]:
@dataclass
class FrameAnalysis:
    """Resultado de un fotograma: nube fusionada, etapas de filtrado y candidatos."""
    frame: LidarFrame
    cloud: FilteredCloud
    candidates: list   # Fragment: caja final y puntos exactos de su clúster

    @property
    def boxes(self):
        return [c.box for c in self.candidates]

    @property
    def groups(self):
        """Puntos de cada caja, alineados con boxes (se dibujan en el vídeo)."""
        return [c.points for c in self.candidates]

    @property
    def count(self):
        return len(self.candidates)


class DetectionPipeline:
    """Fusión -> filtrado -> detección de cada fotograma, de forma independiente (sin tracking)."""

    def __init__(self, sequence=DEMO_SEQUENCE, config=DETECTOR_CONFIG):
        # Una configuración parcial se completa con los valores por defecto del detector.
        config = {**DETECTOR_DEFAULTS, **config}
        self.dataset = SequenceDataset(sequence)
        self.road = RoadMap(MAP_FILE)
        self.filter = PointCloudFilter(self.road, config['min_height'])
        self.detector = VehicleDetector(self.road, config)

    def run(self, indices=None):
        for index in (range(len(self.dataset)) if indices is None else indices):
            yield self.analyze(index)

    def analyze(self, index):
        return self._process(self.dataset.frame(index))

    def _process(self, frame):
        cloud = self.filter.apply(frame.points)
        return FrameAnalysis(frame, cloud, self.detector.detect(cloud))

## Lectura de resultados

In [ ]:
@dataclass
class SequenceResults:
    """Detecciones por fotograma de una secuencia (formato de main.py)."""
    name: str
    records: list

    @classmethod
    def load(cls, name, root=RESULTS_DIR):
        path = Path(root) / name / 'detections.jsonl'
        if not path.is_file():
            raise FileNotFoundError(f'Falta {path}: ejecutar antes la sección 0 (preparación)')
        records = [json.loads(line) for line in path.read_text().splitlines()]
        if len(records) != FRAMES_PER_SEQUENCE:
            raise ValueError(f'Se esperan {FRAMES_PER_SEQUENCE} fotogramas en {name}')
        if any(r['count'] != len(r['boxes']) for r in records):
            raise ValueError(f'El conteo no coincide con las cajas en {name}')
        return cls(name, records)

    def __len__(self):
        return len(self.records)

    @property
    def counts(self):
        return [r['count'] for r in self.records]

    @property
    def times_s(self):
        start = self.records[0]['timestamp_ms']
        return [(r['timestamp_ms'] - start) / 1000 for r in self.records]

    def count_at(self, frame_index):
        return next(r['count'] for r in self.records if r['frame_index'] == frame_index)

## Visualización

In [ ]:
class SceneVisualizer:
    """Tablas y figuras cenitales de la presentación."""
    MAX_POINTS = 50000      # submuestreo aproximado de las nubes dibujadas
    SENSOR_VOXEL = 0.45     # vóxel solo para dibujar cada LiDAR por separado
    TH_STYLE = 'text-align:left;padding:8px'
    TD_STYLE = 'padding:8px;border-bottom:1px solid #ddd'

    def __init__(self, road, roi=ROI):
        self.road = road
        self.roi = roi

    @classmethod
    def table(cls, headers, rows):
        """Tabla HTML sencilla con los valores escapados."""
        header = cls._cells('th', cls.TH_STYLE, headers)
        body = ''.join('<tr>' + cls._cells('td', cls.TD_STYLE, row) + '</tr>' for row in rows)
        display(HTML('<table style="border-collapse:collapse"><thead><tr>' + header
                     + '</tr></thead><tbody>' + body + '</tbody></table>'))

    @staticmethod
    def _cells(tag, style, values):
        return ''.join(f'<{tag} style="{style}">{html.escape(str(v))}</{tag}>' for v in values)

    def show_fusion(self, frame):
        self.table(['Sensor', 'Archivo sincronizado'], frame.sources.items())
        print(f'Fotograma {frame.index} · timestamp {frame.timestamp_ms} ms · '
              f'{len(frame.points):,} puntos fusionados')
        fig, ax = plt.subplots(figsize=(10, 8))
        for sensor, cloud in frame.clouds.items():
            xyz = PointCloudFilter.crop(cloud, self.roi)
            xyz = PointCloudFilter.voxelize(xyz, self.SENSOR_VOXEL)
            ax.scatter(xyz[:, 0], xyz[:, 1], s=0.7, alpha=0.6, label=sensor)
        self._bev_axes(ax, 'Cuatro puntos de vista en un sistema común',
                       'X global (m)', 'Y global (m)')
        ax.legend(markerscale=5, fontsize=8, loc='upper left')
        self._show(fig)

    def show_filtering(self, cloud):
        self.table(['Etapa', 'Puntos'], [('Dentro de la ROI', len(cloud.cropped)),
                                         ('Tras vóxeles', len(cloud.reduced)),
                                         ('Elevados cerca de calzada', len(cloud.foreground))])
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        titles = ['Nube reducida y máscara vial', 'Puntos para el agrupamiento']
        for ax, points, title in zip(axes, [cloud.reduced, cloud.foreground], titles):
            sample = self._sample(points)
            ax.scatter(sample[:, 0], sample[:, 1], s=0.6, color='#536878')
            self.road.draw(ax)
            self._bev_axes(ax, title)
        self._show(fig)

    def show_detection(self, analysis):
        boxes = analysis.boxes
        print(f'Conteo estimado en este fotograma: {len(boxes)} candidatos a vehículo')
        self.table(['Caja local', 'Centro XYZ (m)', 'Largo × ancho × alto (m)', 'Giro (rad)'],
                   [self._box_row(i, b) for i, b in enumerate(boxes[:6], 1)])
        print('Se muestran las primeras seis cajas; los índices no son identidades de tracking.')
        fig, ax = plt.subplots(figsize=(10, 8))
        sample = self._sample(analysis.cloud.reduced)
        ax.scatter(sample[:, 0], sample[:, 1], s=0.5, color='0.7')
        self.road.draw(ax)
        for i, b in enumerate(boxes, 1):
            ax.plot(*BoxFitter.footprint(b).exterior.xy, color='crimson', linewidth=1.1)
            ax.text(*b['center'][:2], str(i), fontsize=7)
        title = f'Detección final: {len(boxes)} cajas · fotograma {analysis.frame.index}'
        self._bev_axes(ax, title)
        self._show(fig)

    def show_counts(self, results, frame=DEMO_FRAME):
        headers = ['Secuencia', 'Fotogramas procesados',
                   f'Vehículos detectados en el fotograma {frame}']
        self.table(headers, [(r.name, len(r), r.count_at(frame)) for r in results])
        fig, axes = plt.subplots(len(results), 1, figsize=(12, 8),
                                 layout='constrained', squeeze=False)
        for ax, r in zip(axes[:, 0], results):
            ax.plot(r.times_s, r.counts, color='#146b9b', linewidth=1.5)
            ax.set(title=r.name, xlabel='Tiempo (s)', ylabel='Vehículos detectados', ylim=(0, None))
            ax.grid(alpha=0.2)
        self._show(fig, tight=False)

    def _sample(self, points):
        return points[::max(1, len(points) // self.MAX_POINTS)]

    def _bev_axes(self, ax, title, xlabel='X (m)', ylabel='Y (m)'):
        ax.set(xlim=self.roi[:2], ylim=self.roi[2:], aspect='equal',
               xlabel=xlabel, ylabel=ylabel, title=title)

    @staticmethod
    def _box_row(index, box):
        return (index, ', '.join(f'{v:.2f}' for v in box['center']),
                ' × '.join(f'{v:.2f}' for v in box['dimensions']), f"{box['yaw_rad']:.2f}")

    @staticmethod
    def _show(fig, tight=True):
        if tight:
            fig.tight_layout()
        display(fig)
        plt.close(fig)

In [ ]:
class VideoGallery:
    """Reproductores HTML de los vídeos exportados, con avance fotograma a fotograma."""

    def __init__(self, folder=VIDEOS_DIR, sequences=SEQUENCES, fps=VIDEO_FPS,
                 frames=FRAMES_PER_SEQUENCE):
        self.folder = Path(folder)
        self.sequences = sequences
        self.fps = fps
        self.frames = frames

    def show(self):
        missing = [name for name in self.sequences if not (self.folder / f'{name}.mp4').is_file()]
        if missing:
            raise FileNotFoundError(f'Faltan vídeos en {self.folder}: {", ".join(missing)}. '
                                    'Ejecutar antes la sección 0 (preparación)')
        display(HTML(self.html()))

    def html(self):
        """Los vídeos se incrustan en base64 para que el notebook sea autocontenido."""
        cards = ''.join(self._card(index, name) for index, name in enumerate(self.sequences))
        return self._intro() + cards + '</div>' + self._script()

    def _source(self, name):
        video = (self.folder / f'{name}.mp4').read_bytes()
        return 'data:video/mp4;base64,' + base64.b64encode(video).decode('ascii')

    def _intro(self):
        return ('<div style="font-family:Arial,sans-serif">\n'
                f'<p>{self.frames} fotogramas por secuencia · {self.fps} fps · {self.frames // self.fps} segundos · sin tracking.</p>\n'
                '<p>Izquierda: nube LiDAR con el fondo original. Derecha: clústeres detectados, cajas y conteo estimado del instante.\n'
                'Puedes pausar, cambiar la velocidad y ampliar a pantalla completa.</p>\n'
                '<p>Todos los clústeres aceptados y sus cajas se muestran en naranja. Se conserva el fondo original: suelo gris y otros puntos elevados azul claro.\n'
                'El color indica detección; no identifica vehículos ni realiza tracking.</p>\n')

    def _card(self, index, name):
        vid = f'lidar-video-{index}'
        return f"""<section style="margin:24px 0">
<h3>{name}</h3>
<video id="{vid}" controls preload="metadata" playsinline style="width:100%;max-width:1400px" src="{self._source(name)}"></video>
<div style="display:flex;gap:12px;align-items:center;margin:8px 0">
<button onclick="stepFrame('{vid}',-1)">← Fotograma</button>
<button onclick="stepFrame('{vid}',1)">Fotograma →</button>
<label>Velocidad <select onchange="document.getElementById('{vid}').playbackRate=Number(this.value)">
<option value="0.25">0,25×</option><option value="0.5">0,5×</option><option value="1" selected>1×</option></select></label>
</div></section>"""

    def _script(self):
        return f"""<script>
function stepFrame(id, direction) {{
 const v=document.getElementById(id); v.pause();
 const frame=Math.floor(v.currentTime*{self.fps}+0.001);
 v.currentTime=(Math.max(0,Math.min({self.frames - 1},frame+direction))+0.01)/{self.fps};
}}
</script>"""

## Descarga y extracción del dataset

In [ ]:
class DatasetDownloader:
    """Secuencias y mapa en data/: reutiliza lo extraído, extrae los .7z o descarga con el devkit.

    download_one_sequence (urbaning) copia en data/ la estructura del repositorio Dataverse:
    dataset/<seq>.7z.00N, labels/<seq>.json y los archivos comunes, mapa lanelet2 incluido.
    """
    DATAVERSE_URL = 'https://dataverse.harvard.edu'   # mismo repositorio que el devkit
    PERSISTENT_ID = 'doi:10.7910/DVN/A9LPY7'
    USER_AGENT = 'pdm-practica1-3'   # Dataverse rechaza (403) el agente por defecto de urllib

    def __init__(self, data_dir=DATA_DIR, map_file=MAP_FILE):
        self.data_dir = Path(data_dir)
        self.dataset_dir = self.data_dir / 'dataset'   # el devkit deja aquí los .7z
        self.map_file = Path(map_file)

    def prepare_sequence(self, name):
        if self.is_extracted(name):
            return 'reutilizados'
        downloaded = not self._first_volume(name).is_file()
        if downloaded:
            self._download(name)
        try:
            self._extract(name)
        except (py7zr.exceptions.ArchiveError, EOFError, OSError):
            if downloaded:
                raise
            # Volúmenes incompletos o dañados: el devkit comprueba su MD5 y descarga los que falten.
            print(f'{name}: archivos .7z incompletos o dañados, se completa la descarga')
            self._download(name)
            self._extract(name)
            downloaded = True
        if not self.is_extracted(name):
            raise RuntimeError(f'La extracción de {name} no contiene los cuatro LiDAR sincronizados')
        return 'descargados y extraídos' if downloaded else 'extraídos'

    def prepare_map(self):
        if self.map_file.is_file():
            return 'reutilizado'
        self._download_map()
        return 'descargado'

    def is_extracted(self, name):
        """Calibración, sincronización y todos los archivos sincronizados de los cuatro LiDAR."""
        folder = self.dataset_dir / name
        sync_file = folder / 'timesync_info.csv'
        if not ((folder / 'calibration.json').is_file() and sync_file.is_file()):
            return False
        sync = SequenceDataset.read_sync(sync_file)
        return all(sensor in sync and all(f and (folder / sensor / f).is_file() for f in sync[sensor])
                   for sensor in SENSORS)

    def _first_volume(self, name):
        return self.dataset_dir / f'{name}.7z.001'

    def _extract(self, name):
        """Igual que extract_dataset.py; una carpeta nueva se extrae con sufijo y se renombra al final."""
        target = self.dataset_dir / name
        # Una carpeta incompleta ya existente se completa en su sitio, como en extract_dataset.py.
        staging = target if target.exists() else target.with_name(f'{name}.partial')
        staging.mkdir(exist_ok=True)
        print(f'{name}: extrayendo los archivos .7z...', flush=True)
        # Los .7z se conservan (como en extract_dataset.py): evitan otra descarga si hay que reextraer.
        with multivolumefile.open(self.dataset_dir / f'{name}.7z', mode='rb') as volumes:
            with py7zr.SevenZipFile(volumes, mode='r') as archive:
                archive.extractall(path=staging)
        if staging != target:
            staging.rename(target)

    def _download(self, name):
        try:
            from urbaning.data import download_one_sequence
        except ImportError as error:
            raise ImportError('La descarga requiere el devkit urbaning: "uv sync --extra notebook"') from error
        print(f'{name}: descargando con el devkit urbaning (varios GB)...', flush=True)
        download_one_sequence(download_dir=self.data_dir.resolve(), sequence_name=name)
        # El devkit descarga con curl sin avisar de los fallos: se comprueba el resultado.
        if not self._first_volume(name).is_file():
            raise RuntimeError(f'No se ha podido descargar {name}: revisar la conexión y reejecutar')

    def _download_map(self):
        """El devkit solo trae el mapa junto a una secuencia: aquí se descarga únicamente el mapa."""
        try:
            from pyDataverse.api import NativeApi   # dependencia del devkit urbaning
        except ImportError as error:
            raise ImportError('La descarga del mapa requiere el devkit urbaning (pyDataverse): '
                              '"uv sync --extra notebook"') from error
        print(f'Descargando {self.map_file.name}...', flush=True)
        files = NativeApi(self.DATAVERSE_URL).get_dataset(self.PERSISTENT_ID).json()
        entry = next((f['dataFile'] for f in files['data']['latestVersion']['files']
                      if f['label'] == self.map_file.name), None)
        if entry is None:
            raise RuntimeError(f'{self.map_file.name} no figura en el repositorio del dataset')
        content = self._fetch(f"{self.DATAVERSE_URL}/api/access/datafile/{entry['id']}")
        if hashlib.md5(content).hexdigest() != entry['md5']:
            raise RuntimeError(f'{self.map_file.name} descargado con errores (MD5): reejecutar')
        pending = self.map_file.with_suffix('.pending' + self.map_file.suffix)
        self.map_file.parent.mkdir(parents=True, exist_ok=True)
        pending.write_bytes(content)
        pending.replace(self.map_file)

    def _fetch(self, url):
        request = urllib.request.Request(url, headers={'User-Agent': self.USER_AGENT})
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                return response.read()
        except (urllib.error.URLError, TimeoutError) as error:
            raise RuntimeError(f'No se ha podido descargar {self.map_file.name} ({error}): '
                               'revisar la conexión y reejecutar') from error

## Exportación de detecciones

In [ ]:
class ResultsExporter:
    """Detecciones en results/improved/<seq>/ con el mismo formato que main.py.

    Así evaluate.py y render_video.py aceptan también los resultados exportados desde aquí.
    """
    FILES = ('road_mask.geojson', 'counts.csv', 'detections.jsonl', 'run.json')   # run.json, el último
    PREVIEW_POINTS = 60000   # submuestreo de la vista PNG, como main.save_preview

    def __init__(self, root=RESULTS_DIR, preview_every=PREVIEW_EVERY):
        self.root = Path(root)
        self.preview_every = preview_every

    def is_complete(self, dataset):
        """Los cuatro archivos y un registro por fotograma, en orden y con los tiempos de la secuencia."""
        folder = self.root / dataset.name
        if not all((folder / name).is_file() for name in self.FILES):
            return False
        try:
            run = json.loads((folder / 'run.json').read_text())
            records = self.load(dataset.name)
            frames = [r['frame_index'] for r in records]
            times = [r['timestamp_ms'] for r in records]
        except (ValueError, KeyError):   # JSON truncado o registros incompletos
            return False
        return (run.get('method') == 'improved' and run.get('tracking') is False
                and set(run.get('sensors', ())) == set(SENSORS)
                and frames == list(range(len(dataset)))
                and np.array_equal(times, dataset.timestamps_ms))

    def is_current(self, name):
        """run.json se generó con la configuración actual del detector, la ROI y el vóxel."""
        try:
            run = json.loads((self.root / name / 'run.json').read_text())
        except (OSError, ValueError):
            return False
        parameters = run.get('parameters', {})
        return (run.get('detector_config') == {**DETECTOR_DEFAULTS, **DETECTOR_CONFIG}
                and parameters.get('roi') == list(ROI) and parameters.get('voxel') == VOXEL_SIZE)

    def load(self, name):
        return SequenceResults.load(name, self.root).records

    def writer(self, pipeline):
        return DetectionWriter(self, pipeline)

    def metadata(self, pipeline):
        """run.json con la estructura de main.py y los valores de su ejecución sin argumentos."""
        config = pipeline.detector.config
        parameters = {'data': self._cli_path(DATA_DIR), 'output': self._cli_path(self.root),
                      'method': 'improved', 'config': str(CONFIG_FILE), 'sequence': None,
                      'start': 0, 'limit': None, 'step': 1, 'roi': list(pipeline.filter.roi),
                      'voxel': pipeline.filter.voxel, 'eps': config['eps'],
                      'min_points': config['min_points'], 'preview_every': self.preview_every}
        return {'sensors': SENSORS, 'coordinate_frame': 'global', 'units': 'm',
                'method': 'improved', 'tracking': False, 'detector_config': config,
                'parameters': parameters}

    @staticmethod
    def road_feature(road):
        return {'type': 'Feature', 'geometry': mapping(road.polygon),
                'properties': {'coordinates': 'global local UTM32, meters; not longitude/latitude',
                               'margin_m': ROAD_MARGIN}}

    @staticmethod
    def record(analysis):
        """Registro JSON de un fotograma, con las claves de main.py."""
        frame = analysis.frame
        return {'frame_index': frame.index, 'timestamp_ms': frame.timestamp_ms,
                'sources': frame.sources, 'count': analysis.count,
                'ground_plane_abc': analysis.cloud.ground.plane.tolist(), 'boxes': analysis.boxes}

    @staticmethod
    def verify(current, saved):
        """Las detecciones recalculadas deben coincidir con las guardadas (como en render_video.py)."""
        frame = saved['frame_index']
        if current['sources'] != saved['sources'] or saved['count'] != len(saved['boxes']):
            raise ValueError(f'Fotograma {frame}: detecciones inconsistentes con los archivos de origen o el conteo')
        same = len(current['boxes']) == len(saved['boxes']) and all(
            a['num_points'] == b['num_points'] and np.allclose(
                a['center'] + a['dimensions'] + [a['yaw_rad']],
                b['center'] + b['dimensions'] + [b['yaw_rad']], atol=1e-6, rtol=0)
            for a, b in zip(current['boxes'], saved['boxes']))
        if not same:
            raise ValueError(f'Fotograma {frame}: el detector no reproduce las detecciones guardadas. '
                             'Borrar la carpeta de resultados de la secuencia para regenerarlas.')

    def save_preview(self, analysis, road, roi, path, title):
        """Vista cenital PNG como main.save_preview (estilo por defecto de matplotlib, sin pyplot)."""
        points, boxes = analysis.cloud.reduced, analysis.boxes
        with plt.style.context('default'):
            fig = Figure(figsize=(10, 10))
            ax = fig.subplots()
            sample = points[::max(1, len(points) // self.PREVIEW_POINTS)]
            ax.scatter(sample[:, 0], sample[:, 1], s=0.3, color='0.5', rasterized=True)
            road.draw(ax)
            for index, box in enumerate(boxes, 1):
                ax.plot(*BoxFitter.footprint(box).exterior.xy, color='tab:red', linewidth=1)
                ax.text(*box['center'][:2], str(index), fontsize=8, color='darkred')
            ax.set(xlim=roi[:2], ylim=roi[2:], xlabel='X global (m)', ylabel='Y global (m)',
                   title=f'{title}\nCandidatos a vehículo: {len(boxes)} · sin tracking', aspect='equal')
            fig.tight_layout()
            fig.savefig(path, dpi=150)

    @staticmethod
    def _cli_path(path):
        """Ruta como la guarda main.py: relativa a la carpeta del proyecto si está dentro."""
        path = Path(path)
        return str(path.relative_to(ROOT) if path.is_relative_to(ROOT) else path)


class DetectionWriter:
    """Archivos de una secuencia en curso: se escriben como '*.pending.*' y se publican al terminar.

    Así una ejecución interrumpida no deja resultados que parezcan completos.
    """

    def __init__(self, exporter, pipeline):
        self.exporter = exporter
        self.pipeline = pipeline
        self.folder = exporter.root / pipeline.dataset.name
        self.pending = {name: self.folder / name.replace('.', '.pending.') for name in exporter.FILES}

    def __enter__(self):
        self.folder.mkdir(parents=True, exist_ok=True)
        metadata = self.exporter.metadata(self.pipeline)
        self.pending['run.json'].write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        road = self.exporter.road_feature(self.pipeline.road)
        self.pending['road_mask.geojson'].write_text(json.dumps(road), encoding='utf-8')
        self.detections = self.pending['detections.jsonl'].open('w', encoding='utf-8')
        self.counts = self.pending['counts.csv'].open('w', newline='', encoding='utf-8')
        self.table = csv.writer(self.counts)
        self.table.writerow(['frame_index', 'timestamp_ms', 'vehicle_candidate_count'])
        return self

    def write(self, analysis, record):
        self.detections.write(json.dumps(record) + '\n')
        self.table.writerow([record['frame_index'], record['timestamp_ms'], record['count']])
        index, every = record['frame_index'], self.exporter.preview_every
        if every and index % every == 0:
            title = f'{self.pipeline.dataset.name} | {record["timestamp_ms"]}'
            self.exporter.save_preview(analysis, self.pipeline.road, self.pipeline.filter.roi,
                                       self.folder / f'bev_{index:04d}.png', title)

    def __exit__(self, kind, error, traceback):
        self.detections.close()
        self.counts.close()
        for name, pending in self.pending.items():
            if kind is None:
                pending.replace(self.folder / name)
            else:
                pending.unlink(missing_ok=True)

## Vídeos de detecciones

In [ ]:
class VideoRenderer:
    """Vídeo cenital de cada secuencia en results/videos/, igual que render_video.py (sin tracking).

    render_video.py guarda en results/videos/clusters los puntos de cada caja para no repetir la
    detección; aquí esos puntos salen de la misma pasada del detector, así que no hace falta caché.
    """
    SIZE = (1800, 1080)          # lienzo (px)
    PANEL = 840                  # lado de cada vista cenital (px)
    TOP = 155
    LEFTS = (40, 920)            # paneles: nube LiDAR | detecciones
    CANVAS_COLOR = (9, 16, 26)
    PANEL_COLOR = (14, 24, 36)
    GROUND_COLOR = (66, 79, 90)
    ELEVATED_COLOR = (130, 213, 236)
    DETECTION_COLOR = (255, 145, 35)
    GROUND_HEIGHT = 0.4          # por debajo, el punto se pinta como suelo (m)
    SNAPSHOT_FRAME = 100         # fotograma guardado también como .jpg
    FONTS = ('C:/Windows/Fonts/arial.ttf', 'DejaVuSans.ttf')

    def __init__(self, folder=VIDEOS_DIR, roi=ROI):
        if not np.isclose(roi[1] - roi[0], roi[3] - roi[2]):
            raise ValueError('La vista cenital requiere ROI cuadrada para conservar las proporciones')
        self.folder = Path(folder)
        self.roi = roi

    def paths(self, name):
        return self.folder / f'{name}.mp4', self.folder / f'{name}.jpg'

    def is_complete(self, name):
        return self.paths(name)[0].is_file()

    def writer(self, name, road, times_ms):
        return VideoWriter(self, name, road, times_ms)

    def render(self, analysis, record, name, elapsed, total, road):
        """Fotograma del vídeo: nube original a la izquierda; clústeres y cajas a la derecha."""
        groups = analysis.groups
        if len(groups) != len(record['boxes']):
            raise ValueError('Debe haber un clúster por caja final')
        background = self._background(analysis.cloud.cropped, np.asarray(record['ground_plane_abc']))
        clustered = self._draw_clusters(background.copy(), groups)
        canvas = Image.new('RGB', self.SIZE, self.CANVAS_COLOR)
        draw = ImageDraw.Draw(canvas)
        self._header(draw, record, name, elapsed, total)
        for left, panel in zip(self.LEFTS, (background, clustered)):
            canvas.paste(Image.fromarray(panel), (left, self.TOP))
            # El mapa ya está recortado a la ROI.
            for ring in road.rings:
                xy = self._pixels(np.asarray(ring.coords)) + [left, self.TOP]
                draw.line([tuple(p) for p in xy], fill='#365970', width=1)
            draw.rectangle((left, self.TOP, left + self.PANEL, self.TOP + self.PANEL), outline='#526474')
        for box in record['boxes']:
            corners = np.asarray(BoxFitter.footprint(box).exterior.coords)
            xy = self._pixels(corners) + [self.LEFTS[1], self.TOP]
            draw.line([tuple(p) for p in xy], fill=self.DETECTION_COLOR, width=2)
        self._footer(draw)
        return canvas

    def _background(self, points, plane):
        """Fondo original: suelo gris y retornos elevados azul claro, sin máscara vial visual."""
        panel = np.full((self.PANEL, self.PANEL, 3), self.PANEL_COLOR, dtype=np.uint8)
        # Solo el plano global guardado, sin la rejilla local (igual que render_video.py).
        height = GroundModel.residual(points, plane)
        elevated = (height >= self.GROUND_HEIGHT) & (height <= FOREGROUND_MAX_HEIGHT)
        for selected, color in ((height < self.GROUND_HEIGHT, self.GROUND_COLOR),
                                (elevated, self.ELEVATED_COLOR)):
            xy = self._pixels(points[selected, :2])
            panel[xy[:, 1], xy[:, 0]] = color
        return panel

    def _draw_clusters(self, panel, groups):
        """Marcadores de 3 x 3 píxeles para ver los retornos a escala de todo el cruce."""
        for group in groups:
            xy = self._pixels(group[:, :2])
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    shifted = xy + [dx, dy]
                    valid = ((shifted >= 0) & (shifted < self.PANEL)).all(axis=1)
                    panel[shifted[valid, 1], shifted[valid, 0]] = self.DETECTION_COLOR
        return panel

    def _header(self, draw, record, name, elapsed, total):
        draw.text((self.LEFTS[0], 20), 'UrbanIng-V2X | Vehículos por fotograma', font=self.font(32), fill='white')
        draw.text((self.LEFTS[0], 67), f'{name}   ·   t = {elapsed:.1f} s   ·   Frame {record["frame_index"]} / {total - 1}',
                  font=self.font(23), fill='#b5c8d7')
        draw.text((self.LEFTS[0], 119), 'NUBE LiDAR · sin colorear clústeres', font=self.font(23), fill='#b5c8d7')
        draw.text((self.LEFTS[1], 119), f'DETECCIONES: {record["count"]} candidatos', font=self.font(25), fill='#ffcd66')

    def _footer(self, draw):
        draw.text((self.LEFTS[0], 1010), '11 · 12 · 31 · 32   |   Solo LiDAR de infraestructura   |   Sin tracking',
                  font=self.font(24), fill='white')
        draw.text((self.LEFTS[0], 1044), 'Naranja: clústeres detectados y cajas · Azul claro: otros puntos elevados · Gris: suelo.',
                  font=self.font(21), fill='#b5c8d7')

    def _pixels(self, xy):
        xmin, xmax, ymin, ymax = self.roi
        return np.column_stack(((xy[:, 0] - xmin) / (xmax - xmin) * (self.PANEL - 1),
                                (ymax - xy[:, 1]) / (ymax - ymin) * (self.PANEL - 1))).round().astype(int)

    @classmethod
    @cache
    def font(cls, size):
        for name in cls.FONTS:
            try:
                return ImageFont.truetype(name, size)
            except OSError:
                pass
        return ImageFont.load_default(size=size)


class VideoWriter:
    """Codificación H.264 en '<seq>.pending.mp4', publicada al terminar (como render_video.py)."""
    ENCODER = dict(codec='libx264', pix_fmt_out='yuv420p', macro_block_size=2,
                   output_params=['-crf', '20', '-movflags', '+faststart'])

    def __init__(self, renderer, name, road, times_ms):
        steps = np.diff(times_ms)
        if not len(steps) or np.any(steps <= 0) or not np.allclose(steps, np.median(steps), atol=1):
            raise ValueError('El vídeo requiere muestreo temporal uniforme')
        self.fps = 1000 / float(np.median(steps))
        self.renderer = renderer
        self.name = name
        self.road = road
        self.times_ms = times_ms
        # Primero la imagen y después el vídeo, que es el que marca la secuencia como completa.
        self.targets = renderer.paths(name)[::-1]
        self.pending = [path.with_suffix('.pending' + path.suffix) for path in self.targets]

    def __enter__(self):
        try:
            import imageio_ffmpeg
        except ImportError as error:
            raise ImportError('Los vídeos requieren imageio-ffmpeg: "uv sync --extra notebook"') from error
        self.renderer.folder.mkdir(parents=True, exist_ok=True)
        self.encoder = imageio_ffmpeg.write_frames(str(self.pending[1]), self.renderer.SIZE,
                                                   fps=self.fps, **self.ENCODER)
        self.encoder.send(None)
        return self

    def write(self, analysis, record):
        index = record['frame_index']
        elapsed = (self.times_ms[index] - self.times_ms[0]) / 1000
        frame = self.renderer.render(analysis, record, self.name, elapsed, len(self.times_ms), self.road)
        self.encoder.send(np.asarray(frame))
        if index == self.renderer.SNAPSHOT_FRAME:
            frame.save(self.pending[0], quality=93)

    def __exit__(self, kind, error, traceback):
        self.encoder.close()
        for pending, target in zip(self.pending, self.targets):
            if kind is None and pending.is_file():
                pending.replace(target)
            else:
                pending.unlink(missing_ok=True)

## Pipeline de preparación

In [ ]:
class PreparationPipeline:
    """Datos -> detecciones -> vídeos: reutiliza lo que ya existe y genera solo lo que falta.

    Si faltan las detecciones y el vídeo de una secuencia, cada fotograma se detecta una sola vez.
    """
    PROGRESS_EVERY = 50   # fotogramas entre mensajes de progreso

    def __init__(self, sequences=SEQUENCES):
        self.sequences = sequences
        self.downloader = DatasetDownloader()
        self.exporter = ResultsExporter()
        self.renderer = VideoRenderer()

    def run(self):
        data = {name: self.downloader.prepare_sequence(name) for name in self.sequences}
        # El mapa llega con la descarga de una secuencia; solo se descarga aparte si aún falta.
        road_map = self.downloader.prepare_map()
        rows = [(name, data[name], *self._prepare_outputs(name)) for name in self.sequences]
        SceneVisualizer.table(['Secuencia', 'Datos', 'Detecciones', 'Vídeo'], rows)
        print(f'Mapa lanelet2: {road_map}.')

    def _prepare_outputs(self, name):
        dataset = SequenceDataset(name)
        has_results = self.exporter.is_complete(dataset)
        has_video = self.renderer.is_complete(name)
        if has_results and not self.exporter.is_current(name):
            # Otra configuración: las detecciones y el vídeo ya no corresponden a los parámetros.
            print(f'{name}: resultados generados con otra configuración del detector; se regeneran')
            has_results = has_video = False
        if has_results and has_video:
            return 'reutilizadas', 'reutilizado'
        saved = self.exporter.load(name) if has_results else None
        task = 'comprobación de detecciones' if has_results else 'detección y exportación'
        print(f'{name}: {task}{"" if has_video else " + vídeo"} ({len(dataset)} fotogramas)', flush=True)
        pipeline = DetectionPipeline(name)
        with ExitStack() as stack:
            results = None if has_results else stack.enter_context(self.exporter.writer(pipeline))
            video = None if has_video else stack.enter_context(
                self.renderer.writer(name, pipeline.road, dataset.timestamps_ms))
            for analysis in pipeline.run():
                record = self.exporter.record(analysis)
                if results is None:
                    # El vídeo dibuja lo guardado, tras comprobar que el detector lo reproduce.
                    self.exporter.verify(record, saved[analysis.frame.index])
                    record = saved[analysis.frame.index]
                else:
                    results.write(analysis, record)
                if video is not None:
                    video.write(analysis, record)
                self._progress(analysis.frame.index + 1, len(dataset))
        return 'reutilizadas' if has_results else 'creadas', 'reutilizado' if has_video else 'creado'

    def _progress(self, done, total):
        if done % self.PROGRESS_EVERY == 0 or done == total:
            print(f'  {done}/{total} fotogramas', flush=True)

## 0. Preparación de datos y resultados

Al ejecutar el notebook se comprueba qué existe ya en disco y solo se genera lo que falta:

1. **Datos** (`data/dataset/<secuencia>/`): si una secuencia no está extraída, se extraen sus archivos `.7z`; si tampoco existen, se descargan antes con el devkit `urbaning` (varios GB por secuencia). También se asegura el mapa lanelet2.
2. **Detecciones** (`results/improved/<secuencia>/`): se procesan todos los fotogramas y se exportan con el mismo formato que `main.py`.
3. **Vídeos** (`results/videos/`): se generan como en `render_video.py`.

Si todo existe, la celda solo comprueba los archivos y termina en pocos segundos.

In [ ]:
PreparationPipeline().run()

## 1. Datos

Secuencias del cruce `crossing1`:

- `20241126_0024_crossing1_09`
- `20241126_0008_crossing1_01`
- `20241127_0000_crossing1_00`

Los sensores 11 y 12 comparten ubicación, pero cada LiDAR tiene su propia calibración. No se utilizan LiDAR de vehículos ni cámaras.

In [ ]:
print('Sensores utilizados:', ', '.join(SENSORS))
print('Asociación entre fotogramas: desactivada (sin tracking).')

## 2. Sincronización y fusión de los cuatro LiDAR

Se seleccionan los archivos del mismo instante mediante `timesync_info.csv` y se transforma cada nube a coordenadas globales con su calibración: **`p_global = R · p_LiDAR + t`**.

Las cuatro nubes se fusionan antes de detectar, para evitar sumar detecciones independientes del mismo vehículo. Se muestra el fotograma 100 de `20241126_0008_crossing1_01`.

In [ ]:
pipeline = DetectionPipeline(DEMO_SEQUENCE)
analysis = pipeline.analyze(DEMO_FRAME)
visualizer = SceneVisualizer(pipeline.road)
visualizer.show_fusion(analysis.frame)

## 3. Filtrado de la nube

Se reduce la densidad de puntos con vóxeles, se estima la altura del suelo y se conservan los puntos elevados próximos a la calzada. El mapa vial delimita la zona de detección.

Los vehículos detenidos también se consideran: no se exige movimiento.

In [ ]:
visualizer.show_filtering(analysis.cloud)

## 4. Detección y bounding boxes

Se agrupan los puntos mediante DBSCAN y se unen fragmentos compatibles de un mismo vehículo. La unión debe cumplir restricciones de orientación, distancia y dimensiones.

Se ajustan cajas orientadas con base en el suelo local. Para completar observaciones parciales se comparan varias posiciones y orientaciones, conservando los puntos observados y penalizando invadir otras cajas. Se suprimen duplicados restantes.

**El conteo es el número de cajas aceptadas en el fotograma.** No se asocian vehículos entre instantes.

In [ ]:
clustering = {k: pipeline.detector.config[k] for k in ('eps', 'min_points')}
print('Parámetros de agrupamiento:', clustering)
visualizer.show_detection(analysis)

### Vídeos de las tres secuencias

Cada vídeo muestra 200 fotogramas a 10 fps. Se conserva el fondo original: suelo gris y puntos elevados azul claro. A la derecha, los puntos de todos los clústeres aceptados y sus cajas aparecen en el mismo naranja. El conteo corresponde a las cajas de cada instante, **sin tracking**. Se pueden pausar y reproducir a menor velocidad.

In [ ]:
VideoGallery().show()

## 5. Conteo en las tres secuencias

Conteos por fotograma obtenidos con los cuatro LiDAR de infraestructura. Cada punto de las gráficas es una detección independiente, **sin tracking**.

Son conteos estimados: pueden persistir falsas detecciones y omisiones. No se suman para obtener vehículos únicos.

In [ ]:
results = [SequenceResults.load(name) for name in SEQUENCES]
visualizer.show_counts(results)